# Transformar Datos Locations
- Eliminar Registros null del compo location_id
- Eliminar registros duplicados
- Eliminar registros duplicados en base a un campo
- convertir columnas en su tipo de dato correcto
- Escribir Datos en la schema silver

In [ ]:
df_locations=spark.sql("SELECT * FROM zenviro.bronze.locations_py")

In [ ]:
df_locations=spark.table("zenviro.bronze.locations_py")

In [ ]:
df_locations=spark.read.table("zenviro.bronze.locations_py")

In [ ]:
display(df_locations)

In [ ]:
df_locations_not_null=df_locations.where("locations_id is not null")

In [ ]:
df_locations_not_null=df_locations.where(df_locations.locations_id.isNotNull())

In [ ]:
df_locations_not_null=df_locations.filter("locations_id is not null")

In [ ]:
df_locations_not_null=df_locations.where(df_locations.locations_id.isNotNull()).orderBy("locations_id")

In [ ]:
df_locations_distinct=df_locations_not_null.distinct()

In [ ]:
df_locations_distinct=df_locations_not_null.dropDuplicates()

In [ ]:
from pyspark.sql.functions import max

In [ ]:
df_locations_max=(
    df_locations_distinct.groupBy("locations_id")
    .agg(max("created_timestamp").alias("max_created_timestamp"))\
    .orderBy("locations_id")
    )
display(df_locations_max)

In [ ]:
location_distinct_tm=df_locations_max.join(df_locations_distinct, 
            df_locations_max.locations_id==df_locations_distinct.locations_id & 
            df_locations_max.created_timestamp==df_locations_distinct.created_timestamp,"inner")\
            .select(df_locations_distinct["*"])

display(location_distinct_tm)

In [ ]:
locations_cast_df=(
location_distinct_tm.select(location_distinct_tm.created_timestamp.cast("timestamp").alias("locations_id"),
                            location_distinct_tm.location_name,
                            location_distinct_tm.address,
                            location_distinct_tm.city,
                            location_distinct_tm.country)
)

In [ ]:
locations_cast_df.writeTo("zenviro.silver.locations_py").createOrReplace()